# Information vs intensity contrast (referee analysis)

The intensities are renormalised so that $\sum_v I_v = 1$, so the total
photon budget is *fixed*: the contrast $r = I_\mathrm{mid}/I_\mathrm{out}$
only **reallocates** a constant budget between the depth (middle) source and
the two transverse width sources. `target_point_source_idx = 0` is the middle
(depth) source, so $r>1$ brightens the depth source at the expense of the
width pair, and $r<1$ does the opposite.

This is the non-uniform analogue of the uniform-efficiency argument in the
paper (a uniform $\eta$ rescales every channel equally and cancels in the
MS-vs-quantum comparison).

In [ ]:
import glob
import re

import matplotlib.pyplot as plt
import numpy as np

from visualize_helpers import latexify

latexify()
FORMAT = "pdf"

In [ ]:
%matplotlib inline

## Load data

In [ ]:
# Read the contrasts from whatever simulation output is on disk, so this
# notebook works for both the original and the denser log-symmetric sweep.
contrasts = sorted(
    float(re.search(r"qfi_(\d+\.\d+)\.npy", f).group(1))
    for f in glob.glob("qfi_*.npy")
    if "_inv_" not in f
)

coords = np.load("coordinates.npy")
p_range = -coords[0, :, 6]
x_range = coords[:, 0, 2]

qfis = [np.load(f"qfi_{c:.2f}.npy") for c in contrasts]
qfis_inv = [np.load(f"qfi_inv_{c:.2f}.npy") for c in contrasts]
fis = [np.load(f"fi_ms_{c:.2f}.npy") for c in contrasts]
fis_inv = [np.load(f"fi_ms_inv_{c:.2f}.npy") for c in contrasts]
contrasts

In [ ]:
# Per-source weights for V = 3 (sum to 1 by construction), and helpers for
# slicing a single geometry out of the (Nx, Nz, 2, 2) bound arrays.
c_arr = np.array(contrasts)
baseline = contrasts.index(1.0)
I_mid = c_arr / (c_arr + 2)  # middle / depth source weight
I_out = 1.0 / (c_arr + 2)  # each transverse / width source weight

fis_sum = [np.sum(fi, axis=2) for fi in fis]  # sum MS FI over mode channels


def _idx(arr: np.ndarray, val: float) -> int:
    """Grid index nearest to a target coordinate value."""
    return int(np.argmin(np.abs(arr - val)))


def _series(arrays: list[np.ndarray], ix: int, iz: int, a: int, b: int) -> np.ndarray:
    """Bound component (a, b) at a fixed geometry, one value per contrast."""
    return np.array([A[ix, iz, a, b] for A in arrays])


# A clean width signal lives at small depth and vice-versa.
iz_w = _idx(p_range, 0.2)
ix_d = _idx(x_range, 0.3)
width_ix = [_idx(x_range, v) for v in (0.5, 1.0, 2.0, 3.5)]
depth_iz = [_idx(p_range, v) for v in (0.2, 0.5, 1.0, 1.7)]

# Single both-sub-Rayleigh geometry for the asymmetry / transfer figures.
ix_sel, iz_sel = _idx(x_range, 0.5 / np.sqrt(2)), _idx(p_range, 0.5)

# Three representative (delta x, delta z) regimes for the FI/CRB-vs-contrast
# figure, in normalized units (sqrt(k/z_R) delta x and delta z / z_R); the
# Rayleigh scale is ~1, so sub-Rayleigh means < 1.
geoms = [
    (
        "both_subR",
        "both sub-Rayleigh",
        _idx(x_range, 0.3 / np.sqrt(2)),
        _idx(p_range, 0.3),
    ),
    (
        "width_subR_depth_R",
        r"width sub-Rayleigh, depth $\approx z_R$",
        _idx(x_range, 0.3 / np.sqrt(2)),
        _idx(p_range, 1.0),
    ),
    (
        "width_R_depth_subR",
        r"width $\approx z_R$, depth sub-Rayleigh",
        _idx(x_range, 1.0 / np.sqrt(2)),
        _idx(p_range, 0.3),
    ),
]
Q, MS = "tab:red", "tab:blue"
geom_txt = (
    rf"$\sqrt{{k/z_R}}\,\delta x={np.sqrt(2) * x_range[ix_sel]:.2f}$, "
    rf"$\delta z/z_R={p_range[iz_sel]:.2f}$"
)

## 1. FI and CRB vs contrast at representative geometries

All three geometry regimes on one figure: QFI in shades of red and
mode-sorting FI in shades of blue, with the shade fixing the $(\delta x,
\delta z)$ pair (light to dark). Panels are arranged rows = parameter
(width $\delta x$ top, depth $\delta z$ bottom), columns = quantity (FI
left, CRB right). The regimes are (i) both parameters sub-Rayleigh, (ii)
width sub-Rayleigh with depth near the Rayleigh length, and (iii) the
reverse -- the table columns give $(\sqrt{k/z_R}\,\delta x,\ \delta
z/z_R)$, where $\approx 1$ is the Rayleigh scale. The dashed vertical line
marks equal brightness ($r = 1$).

In [ ]:
# Shade encodes the (delta x, delta z) pair; hue encodes QFI (red) vs MS (blue).
shades_q = plt.cm.Reds(np.linspace(0.45, 0.95, len(geoms)))
shades_m = plt.cm.Blues(np.linspace(0.45, 0.95, len(geoms)))

# Panels: rows = parameter (width / depth), columns = quantity (FI / CRB).
panel_spec = [
    (0, 0, qfis, fis_sum, r"$H_{\delta x \delta x}$"),
    (0, 0, qfis_inv, fis_inv, r"$H^{-1}_{\delta x \delta x}$"),
    (1, 1, qfis, fis_sum, r"$H_{\delta z \delta z}$"),
    (1, 1, qfis_inv, fis_inv, r"$H^{-1}_{\delta z \delta z}$"),
]
fig, axes = plt.subplots(2, 2, figsize=(15, 13))
for ax, (a, b, q_arrs, ms_arrs, ysym) in zip(axes.flat, panel_spec):
    for i, (_tag, _glabel, ix, iz) in enumerate(geoms):
        ax.plot(c_arr, _series(q_arrs, ix, iz, a, b), "o-", color=shades_q[i])
        ax.plot(c_arr, _series(ms_arrs, ix, iz, a, b), "s--", color=shades_m[i])
    ax.axvline(1.0, color="0.6", ls=":", lw=1.5)
    ax.set_ylabel(ysym)
    ax.set_xlabel(r"$r_c$")
# Open a central band between the panels to host the legend.
fig.subplots_adjust(
    left=0.11, right=0.97, top=0.96, bottom=0.07, hspace=0.55, wspace=0.30
)

# Table-style legend (borderless): top-left names the tuple, columns are the
# (delta x, delta z) pairs, rows are QFI / MS drawn as thick lines matching the
# curve styles (solid+circle QFI, dashed+square MS) in the per-pair shade.
pair_labels = [
    rf"$({np.sqrt(2) * x_range[ix]:.2f},\ {p_range[iz]:.2f})$" for _, _, ix, iz in geoms
]
lax = fig.add_axes((0.28, 0.455, 0.56, 0.085))
lax.set_xlim(0, 1)
lax.set_ylim(0, 1)
lax.axis("off")
lx = 0.10
cx = [0.30 + i * 0.23 for i in range(len(geoms))]
y_hdr, y_q, y_m = 0.82, 0.50, 0.16
lax.text(lx, y_hdr, r"$(\delta x, \delta z)$", ha="center", va="center", fontsize=15)
lax.text(lx, y_q, "QFI", ha="center", va="center", fontsize=15)
lax.text(lx, y_m, "MS", ha="center", va="center", fontsize=15)
for i, x in enumerate(cx):
    lax.text(x, y_hdr, pair_labels[i], ha="center", va="center", fontsize=14)
    lax.plot(
        [x - 0.06, x, x + 0.06],
        [y_q] * 3,
        "-",
        color=shades_q[i],
        lw=4,
        marker="o",
        markersize=10,
        markevery=[1],
        clip_on=False,
    )
    lax.plot(
        [x - 0.06, x, x + 0.06],
        [y_m] * 3,
        "--",
        color=shades_m[i],
        lw=4,
        marker="s",
        markersize=10,
        markevery=[1],
        clip_on=False,
    )

fig.savefig(f"fi_crb_vs_contrast.{FORMAT}", format=FORMAT, bbox_inches="tight")
plt.show()

## 2. Is it only photon reallocation? (normalised collapse)

Each curve is the bound at one geometry, divided by its equal-brightness
value, vs contrast. The dashed black line is the *pure-reallocation*
prediction -- the responsible source's weight, normalised the same way
($I_\mathrm{out}$ for width, $I_\mathrm{mid}$ for depth). Curves landing on
the dashed line are explained entirely by the photon budget moving between
sources; departures are the genuine structural (overlap) effect.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11), sharex=True)
rows = [("QFI", qfis), ("MS", fis_sum)]
w_pred = I_out / I_out[baseline]
d_pred = I_mid / I_mid[baseline]
for r, (name, arrs) in enumerate(rows):
    for ix in width_ix:
        y = _series(arrs, ix, iz_w, 0, 0)
        axes[r, 0].plot(
            c_arr,
            y / y[baseline],
            "-",
            alpha=0.8,
            label=rf"$\sqrt{{k/z_R}}\delta x={np.sqrt(2) * x_range[ix]:.1f}$",
        )
    for iz in depth_iz:
        y = _series(arrs, ix_d, iz, 1, 1)
        axes[r, 1].plot(
            c_arr,
            y / y[baseline],
            "-",
            alpha=0.8,
            label=rf"$\delta z/z_R={p_range[iz]:.1f}$",
        )
    axes[r, 0].plot(c_arr, w_pred, "k--", lw=2.5, label=r"$\propto I_\mathrm{out}$")
    axes[r, 1].plot(c_arr, d_pred, "k--", lw=2.5, label=r"$\propto I_\mathrm{mid}$")
    axes[r, 0].set_ylabel(rf"{name}$(r)\,/\,${name}$(1)$")
axes[0, 0].set_title(r"width $H_{\delta x \delta x}$")
axes[0, 1].set_title(r"depth $H_{\delta z \delta z}$")
for ax in axes[1, :]:
    ax.set_xlabel(r"contrast $r = I_\mathrm{mid}/I_\mathrm{out}$")
axes[0, 0].legend(frameon=False, fontsize=14, ncol=2)
axes[0, 1].legend(frameon=False, fontsize=14, ncol=2)
fig.tight_layout()
fig.savefig(
    f"contrast_reallocation_collapse.{FORMAT}", format=FORMAT, bbox_inches="tight"
)
plt.show()

## 3. Mode-sorting efficiency vs contrast

The ratio $\eta = H_\mathrm{MS}/\mathcal{Q}$ of mode-sorting FI to the
quantum bound. If $\eta$ is flat in $r$, the MS-vs-quantum comparison is
robust to brightness imbalance -- imbalance moves information between the two
parameters but does not change how well mode sorting captures it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ix in width_ix:
    eta = _series(fis_sum, ix, iz_w, 0, 0) / _series(qfis, ix, iz_w, 0, 0)
    axes[0].plot(
        c_arr,
        eta,
        "o-",
        label=rf"$\sqrt{{k/z_R}}\delta x={np.sqrt(2) * x_range[ix]:.1f}$",
    )
for iz in depth_iz:
    eta = _series(fis_sum, ix_d, iz, 1, 1) / _series(qfis, ix_d, iz, 1, 1)
    axes[1].plot(c_arr, eta, "o-", label=rf"$\delta z/z_R={p_range[iz]:.1f}$")
axes[0].set_title(r"width $\eta = H_{\delta x\delta x}/\mathcal{Q}_{\delta x\delta x}$")
axes[1].set_title(r"depth $\eta = H_{\delta z\delta z}/\mathcal{Q}_{\delta z\delta z}$")
for ax in axes:
    ax.axhline(1.0, color="0.6", ls=":", lw=1.5)
    ax.set_xlabel(r"contrast $r = I_\mathrm{mid}/I_\mathrm{out}$")
    ax.set_ylim(0, 1.08)
    ax.legend(frameon=False, fontsize=14)
fig.tight_layout()
fig.savefig(f"ms_efficiency_vs_contrast.{FORMAT}", format=FORMAT, bbox_inches="tight")
plt.show()

## 4. The contrast asymmetry is the weight map, not physics

Width and depth QFI move in opposite directions with $r$ and the response is
asymmetric about $r = 1$ (left), because the source weight $I(r)=r/(r+2)$ is
nonlinear. Re-plotting each against the *responsible source's weight* (right)
removes the asymmetry: the QFI is, to leading order, linear in that weight.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
qxx = _series(qfis, ix_sel, iz_sel, 0, 0)
qzz = _series(qfis, ix_sel, iz_sel, 1, 1)

axes[0].plot(c_arr, qxx, "o-", color=Q, label=r"width $\mathcal{Q}_{\delta x\delta x}$")
axes[0].plot(
    c_arr, qzz, "s-", color=MS, label=r"depth $\mathcal{Q}_{\delta z\delta z}$"
)
axes[0].axvline(1.0, color="0.6", ls=":", lw=1.5)
axes[0].set_xlabel(r"contrast $r = I_\mathrm{mid}/I_\mathrm{out}$")
axes[0].set_ylabel("QFI")
axes[0].set_title("vs contrast (asymmetric)")
axes[0].legend(frameon=False, fontsize=16)

order_w, order_d = np.argsort(I_out), np.argsort(I_mid)
axes[1].plot(
    I_out[order_w], qxx[order_w], "o-", color=Q, label=r"width vs $I_\mathrm{out}$"
)
axes[1].plot(
    I_mid[order_d], qzz[order_d], "s-", color=MS, label=r"depth vs $I_\mathrm{mid}$"
)
axes[1].set_xlabel(r"responsible source weight $I$")
axes[1].set_ylabel("QFI")
axes[1].set_title("vs source weight (linearised)")
axes[1].legend(frameon=False, fontsize=16)
fig.tight_layout()
fig.savefig(f"contrast_asymmetry.{FORMAT}", format=FORMAT, bbox_inches="tight")
plt.show()

## 5. Information transfer: axial ↔ planar brightness

Parametric trade-off at the fixed geometry: each point is one contrast, the
axes are the width ($\delta x$) and depth ($\delta z$) information. As the
contrast switches brightness from the planar (width) sources toward the axial
(depth) source, the curve moves up-and-left -- depth information is *gained*
exactly as width information is *lost*, and vice-versa. The star marks equal
brightness ($r = 1$); QFI and MS trace the same exchange.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, arrs, name in [(axes[0], qfis, "QFI"), (axes[1], fis_sum, "MS")]:
    wx = _series(arrs, ix_sel, iz_sel, 0, 0)
    dz = _series(arrs, ix_sel, iz_sel, 1, 1)
    color = Q if name == "QFI" else MS
    ax.plot(wx, dz, "o-", color=color)
    ax.plot(
        wx[baseline],
        dz[baseline],
        "*",
        color="k",
        ms=22,
        zorder=5,
        label=r"$r = 1$ (equal)",
    )
    ax.annotate(
        f"$r={c_arr[0]:.1f}$\n(planar bright)",
        (wx[0], dz[0]),
        textcoords="offset points",
        xytext=(10, -45),
        fontsize=15,
        ha="left",
    )
    ax.annotate(
        f"$r={c_arr[-1]:.1f}$\n(axial bright)",
        (wx[-1], dz[-1]),
        textcoords="offset points",
        xytext=(-15, 15),
        fontsize=15,
        ha="right",
    )
    ax.set_xlabel(r"width information $\,\delta x \delta x$")
    ax.set_ylabel(r"depth information $\,\delta z \delta z$")
    ax.set_title(name)
    ax.legend(frameon=False, fontsize=16, loc="upper right")
fig.suptitle(f"Width ↔ depth information exchange ({geom_txt})", y=1.0)
fig.tight_layout()
fig.savefig(
    f"contrast_information_transfer.{FORMAT}", format=FORMAT, bbox_inches="tight"
)
plt.show()